# **Phase 4A: Process / Technology Suitability Rules**
---
**Objective:** To assign process suitability technology (Combustion /
Gasification / Pyrolysis) as ranking recommendations (primary, secondary, and
constraint level) from engineered features (not ML
prediction).

In [1]:
# load data
import pandas as pd

df = pd.read_csv("../data/interim/engineered_features.csv")

# importing "cluster" from cluster_df
cluster_df = pd.read_csv("../results/tables/clustering/clustered_with_labels.csv")

# Keep only required columns for merging
cluster_min = cluster_df[["Sample_ID", "Cluster"]]

# Merge on Sample_ID
df = df.merge(
    cluster_min,
    on="Sample_ID",
    how="left"   
)

In [2]:
# Saving files
df.to_csv(
    "../data/interim/engineered_features_with_cluster.csv",
    index=False
)

In [3]:
# Apply suitability rules
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.models.suitability_rules import apply_process_rules

df = apply_process_rules(df)
df[["Primary_Process", "Secondary_Process", "Constraint_Level"]].value_counts()

Primary_Process  Secondary_Process  Constraint_Level
Pyrolysis        Combustion         Low                 13
Gasification     Combustion         Low                  1
Name: count, dtype: int64

In [4]:
df.head()

,Sample_ID,Biomass_Type,Class,Subclass,Ash_db,VM_db,FC_db,C_db,H_db,N_db,...,Alkali_Index,Silica_Ratio,Base_Acid_Ratio,Moist_ar,Moisture_Penalty,Effective_HHV,Cluster,Primary_Process,Secondary_Process,Constraint_Level
0,1,Industrial Processing,Timber industry,Woodchips (Softwood),1.1,81.0,17.9,48.6,6.26,0.20,...,4.99,0.428990,1.975996,41.2,0.023697,-826.110,1.0,Pyrolysis,None,High
1,3,Agricultural,Animal farming,Chicken manure pellets,32.7,56.2,11.1,30.0,3.91,3.94,...,9.03,0.577461,1.623719,18.9,0.050251,-222.139,2.0,Pre-treatment Required,None,Low
2,4,Urban Waste,Biosolids,Treated biosolids,42.8,52.0,5.2,24.7,4.35,4.61,...,1.72,6.864608,0.717266,8.1,0.109890,-89.886,2.0,Pre-treatment Required,None,High
3,5,Industrial Processing,Paper industry,Paper sludge,26.2,64.2,9.6,32.4,4.96,0.47,...,0.82,1.640852,0.404991,7.8,0.113636,-96.152,2.0,Pre-treatment Required,None,Low
4,6,Industrial Processing,Cotton Industry,Cotton seed hulls,1.9,77.9,20.2,32.5,6.02,0.45,...,37.97,0.331797,2.703448,11.6,0.079365,-193.768,1.0,Pyrolysis,None,Moderate


In [5]:
# link suitability with clusters
pd.crosstab(
    df["Cluster"],
    df["Primary_Process"],
    normalize="index"
)

Primary_Process,Gasification,Pre-treatment Required,Pyrolysis
Cluster,,,
0.0,0.0,0.0,1.0
1.0,0.0,0.0,1.0
2.0,0.1,0.8,0.1


In [6]:
# save process suitability by clusters file
df.to_csv(
    "../results/tables/process_suitability_summary.csv",
    index=False
)